# Sesión 08 - Lab 1: Job multi-tarea con dependencias, reintentos y branching

Este notebook contiene la lógica de las 4 tareas del Job que se arma en la Sección 4 del canvas: `ingesta_bronze`, `cuarentena_registros`, `transformar_silver` y `agregacion_gold`. Un mismo notebook puede orquestar varias tareas de un Job real si cada tarea le pasa un **base parameter** distinto: acá ese parámetro se llama `tarea`, y cada bloque de código de más abajo solo corre si `tarea` coincide con su nombre. Así, en el Job real, cada una de las 4 tareas apunta al mismo archivo `sesion08_lab1.ipynb`, pero cada una con un valor distinto de `tarea` en **Parameters**.

Para probarlo de forma interactiva (fuera del Job), cambiá el valor del widget `tarea` de la celda de configuración y volvé a correr el notebook completo (`Run All`); cada vuelta ejecuta una sola tarea, igual que lo haría el Job.

La tarea de validación de calidad (`validar_calidad`) no tiene código: es una tarea de tipo `If/else condition`, configurada directo en la UI del Job (se documenta en el Lab 1B).

## Verificación del entorno

In [0]:
dbutils.fs.ls("/Volumes/dbassociate/default/vol_landing/sesion_08")

## Configuración de la tarea a ejecutar

In [0]:
# un widget es un input que se muestra en la UI del job, es la forma en la que 
dbutils.widgets.dropdown( # dropdown indica que se va a mostrar un dropdown
    "tarea", # nombre del widget
    "ingesta_bronze", # opción por defecto
    ["ingesta_bronze", "cuarentena_registros", "transformar_silver", "agregacion_gold"], # lista de opciones
)
dbutils.widgets.text("forzar_fallo", "false") # el widget text muestra un input de texto con el valor por defecto

# con get podemos obtener el valor del widget
tarea = dbutils.widgets.get("tarea")
forzar_fallo = dbutils.widgets.get("forzar_fallo")

print(f"Ejecutando bloque de la tarea: {tarea}")

## Lab 1A — TAREA `ingesta_bronze`

Lee `pedidos_diarios.csv` con schema explícito, aterriza en Bronze con las columnas de auditoría de siempre, y deja dos valores disponibles para el resto del Job vía `dbutils.jobs.taskValues.set()`: el total de registros y cuántos traen `monto_total` nulo. Este último es el que va a leer la tarea `validar_calidad` (Lab 1B) para decidir el branch.

In [0]:
if tarea == "ingesta_bronze":
    from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType
    from pyspark.sql.functions import current_timestamp, col, lit

    schema_pedidos = StructType([
        StructField("pedido_id", StringType(), False),
        StructField("cliente_id", StringType(), True),
        StructField("fecha_pedido", DateType(), True),
        StructField("canal", StringType(), True),
        StructField("monto_total", DoubleType(), True),
        StructField("estado", StringType(), True),
    ])

    df_raw = (
        spark.read.format("csv")
        .option("header", "true")
        .schema(schema_pedidos)
        .load("/Volumes/dbassociate/default/vol_landing/sesion_08/pedidos_diarios.csv")
    )

    df_bronze = (
        df_raw
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", col("_metadata.file_name"))
        .withColumn("source_system", lit("erp_pedidos"))
        .withColumn("batch_id", lit("sesion08_lab1"))
    )

    df_bronze.write.mode("overwrite").saveAsTable("dbassociate.bronze.pedidos_orquestacion")

    total_registros = df_bronze.count()
    registros_invalidos = df_bronze.filter(col("monto_total").isNull()).count()

    print(f"Total registros cargados en Bronze: {total_registros}")
    print(f"Registros con monto_total nulo: {registros_invalidos}")

    # Estabele un valor de salida, esto puede se tomado por el siguiente task.
    dbutils.jobs.taskValues.set(key="total_registros", value=total_registros)
    dbutils.jobs.taskValues.set(key="registros_invalidos", value=registros_invalidos)

## Lab 1B — TAREA `validar_calidad` (If/else condition, sin código)

Esta tarea se agrega directo en la UI del Job como tipo **If/else condition**, no como Notebook: no tiene celda propia acá. Evalúa:

```
{{tasks.ingesta_bronze.values.registros_invalidos}} > 0
```

- **True** → dispara la tarea `cuarentena_registros` (Depends on: `validar_calidad (true)`).
- **False** → dispara la tarea `transformar_silver` (Depends on: `validar_calidad (false)`).

`pedidos_diarios.csv` trae 4 filas a propósito con `monto_total` nulo, así que en una corrida normal siempre se toma el branch `true`. Para ver el branch `false` en vivo, comentá esas 4 filas del CSV (o subí una copia sin nulos a la misma ruta del Volume) y volvé a correr desde `ingesta_bronze`.

## Lab 1C — TAREA `cuarentena_registros` (branch `true`)

Separa las filas con `monto_total` nulo a una tabla de cuarentena aparte y deja las filas válidas listas en Silver. Incluye el interruptor `forzar_fallo`, usado en el Lab 1F para practicar Repair Run: con `forzar_fallo=true`, la tarea falla a propósito antes de escribir nada.

In [0]:
# Forzamos a que se muera el notebook
if tarea == "cuarentena_registros":
    if forzar_fallo == "true":
        raise RuntimeError(
            "Fallo forzado para practicar Repair Run. "
            "Volvé a poner el parámetro 'forzar_fallo' en 'false' y reparalo desde la UI del Job."
        )

    from pyspark.sql.functions import col

    df_bronze = spark.table("dbassociate.bronze.pedidos_orquestacion")

    df_cuarentena = df_bronze.filter(col("monto_total").isNull())
    df_validos = df_bronze.filter(col("monto_total").isNotNull())

    df_cuarentena.write.mode("overwrite").saveAsTable("dbassociate.bronze.pedidos_orquestacion_cuarentena")
    (
        df_validos.drop("source_file", "batch_id")
        .write.mode("overwrite")
        .saveAsTable("dbassociate.silver.pedidos_orquestacion")
    )

    registros_cuarentena = df_cuarentena.count()
    print(f"Registros en cuarentena (monto_total nulo): {registros_cuarentena}")
    print(f"Registros válidos que pasan a Silver: {df_validos.count()}")

    dbutils.jobs.taskValues.set(key="registros_cuarentena", value=registros_cuarentena)

## Lab 1D — TAREA `transformar_silver` (branch `false`)

Camino directo cuando no hay ningún `monto_total` nulo: todas las filas de Bronze pasan a Silver sin necesidad de separar cuarentena. Con el dataset de este lab no se ejecuta en una corrida normal (ver nota del Lab 1B); queda documentado para cuando se prueba el branch `false`.

In [0]:
if tarea == "transformar_silver":
    df_bronze = spark.table("dbassociate.bronze.pedidos_orquestacion")

    (
        df_bronze.drop("source_file", "batch_id")
        .write.mode("overwrite")
        .saveAsTable("dbassociate.silver.pedidos_orquestacion")
    )

    print(f"Silver generado sin registros en cuarentena: {df_bronze.count()} filas")

## Lab 1E — TAREA `agregacion_gold`

Converge después de cualquiera de los dos branches: solo uno de `cuarentena_registros`/`transformar_silver` corrió, el otro quedó en estado *Skipped* (no *Failed*), así que esta tarea depende de ambos con **Run if dependencies = At least one succeeded**: el detalle de esa configuración va en el Lab 1G.

In [0]:
if tarea == "agregacion_gold":
    from pyspark.sql.functions import count, sum as spark_sum, avg, round as spark_round

    df_silver = spark.table("dbassociate.silver.pedidos_orquestacion")

    df_gold = (
        df_silver.groupBy("canal", "estado")
        .agg(
            count("pedido_id").alias("num_pedidos"),
            spark_round(spark_sum("monto_total"), 2).alias("monto_total_canal"),
            spark_round(avg("monto_total"), 2).alias("ticket_promedio"),
        )
    )

    df_gold.write.mode("overwrite").saveAsTable("dbassociate.gold.pedidos_orquestacion_resumen")

    display(df_gold.orderBy("canal", "estado"))

## Lab 1F — Simular un fallo y repararlo con Repair Run

1. Corré el Job completo una vez con `forzar_fallo=false` en todas las tareas (ver Lab 1G): debería terminar en verde.
2. Editá la tarea `cuarentena_registros`: en **Parameters**, cambiá `forzar_fallo` a `true`. Lanzá **Run now**.
3. `ingesta_bronze` y `validar_calidad` terminan bien; `cuarentena_registros` falla con el `RuntimeError` a propósito; `agregacion_gold` queda *Upstream failed* (no corre).
4. Volvé a editar `cuarentena_registros` y poné `forzar_fallo=false`.
5. Abrí el run fallido → **Repair run**. Databricks vuelve a correr solo `cuarentena_registros` y todo lo que dependía de ella (`agregacion_gold`); no repite `ingesta_bronze` ni `validar_calidad`, que ya habían salido bien.

**Ojo con la idempotencia:** un Repair Run vuelve a correr la tarea fallida desde el principio, no desde donde se cortó. Acá es seguro porque `cuarentena_registros` escribe con `mode("overwrite")`. Si en cambio usara `mode("append")`, un Repair Run después de un fallo parcial podría duplicar filas que sí llegaron a escribirse antes de que la tarea fallara.

## Lab 1G — Construir el Job en la Jobs UI

**Workflows → Jobs & Pipelines → Create Job.** Nombre: `sesion08_pipeline_pedidos`.

**Tarea 1 — `ingesta_bronze`**
- Type: `Notebook` · Source: `Workspace`, este notebook (`sesion08_lab1.ipynb`)
- Parameters: `tarea=ingesta_bronze`
- Compute: Serverless
- Advanced → Retries: 2 (una lectura del Volume puede fallar por una razón transitoria; reintentar antes de escalar tiene sentido acá, no en toda tarea)

**Tarea 2 — `validar_calidad`**
- Type: `If/else condition` · Depends on: `ingesta_bronze`
- Condition: `{{tasks.ingesta_bronze.values.registros_invalidos}}` `>` `0`

**Tarea 3 — `cuarentena_registros`**
- Type: `Notebook` · Source: mismo notebook · Depends on: `validar_calidad (true)`
- Parameters: `tarea=cuarentena_registros`, `forzar_fallo=false`
- Compute: Serverless

**Tarea 4 — `transformar_silver`**
- Type: `Notebook` · Source: mismo notebook · Depends on: `validar_calidad (false)`
- Parameters: `tarea=transformar_silver`
- Compute: Serverless

**Tarea 5 — `agregacion_gold`**
- Type: `Notebook` · Source: mismo notebook
- Depends on: `cuarentena_registros` y `transformar_silver`
- Run if dependencies: **At least one succeeded**
- Parameters: `tarea=agregacion_gold`
- Compute: Serverless

**Create.** Click **Run now** y observar el DAG: `cuarentena_registros` se pone verde, `transformar_silver` queda *Skipped* (gris, no rojo), y `agregacion_gold` corre igual porque al menos una de sus dependencias tuvo éxito.

## Limpieza

In [0]:
# spark.sql("DROP TABLE IF EXISTS dbassociate.gold.pedidos_orquestacion_resumen")
# spark.sql("DROP TABLE IF EXISTS dbassociate.silver.pedidos_orquestacion")
# spark.sql("DROP TABLE IF EXISTS dbassociate.bronze.pedidos_orquestacion_cuarentena")
# spark.sql("DROP TABLE IF EXISTS dbassociate.bronze.pedidos_orquestacion")

# print("Tablas de este laboratorio eliminadas.")